# **Libraries Data Acquisition**

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
path = kagglehub.dataset_download("palbha/cmapss-jet-engine-simulated-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\kisha\.cache\kagglehub\datasets\palbha\cmapss-jet-engine-simulated-data\versions\1


Data sets consists of multiple multivariate time series. Each data set is further divided into training and test subsets. Each time series is from a different engine � i.e., the data can be considered to be from a fleet of engines of the same type. Each engine starts with different degrees of initial wear and manufacturing variation which is unknown to the user. This wear and variation is considered normal, i.e., it is not considered a fault condition. There are three operational settings that have a substantial effect on engine performance. These settings are also included in the data. The data is contaminated with sensor noise.

The engine is operating normally at the start of each time series, and develops a fault at some point during the series. In the training set, the fault grows in magnitude until system failure. In the test set, the time series ends some time prior to system failure. The objective of the competition is to predict the number of remaining operational cycles before failure in the test set, i.e., the number of operational cycles after the last cycle that the engine will continue to operate. Also provided a vector of true Remaining Useful Life (RUL) values for the test data.

In [3]:
print("Available files:", os.listdir(path))
file_path_train = os.path.join(path, "train_FD001.txt")
file_path_test = os.path.join(path, "test_FD001.txt")

col_names = ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
[f'sensor_{i}' for i in range(1, 22)]

df = pd.read_csv(file_path_train, sep=r'\s+', header=None, names=col_names)
test_df = pd.read_csv(file_path_test, sep=r'\s+', header=None, names=col_names)


Available files: ['Damage Propagation Modeling.pdf', 'readme.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']


In [4]:
df.shape

(20631, 26)

In [5]:
test_df.shape


(13096, 26)

In [6]:
df

,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640


# **Data Cleaning & Exploration**

Check for null values

In [7]:
df.isnull().sum()

unit_number       0
time_in_cycles    0
op_setting_1      0
op_setting_2      0
op_setting_3      0
sensor_1          0
sensor_2          0
sensor_3          0
sensor_4          0
sensor_5          0
sensor_6          0
sensor_7          0
sensor_8          0
sensor_9          0
sensor_10         0
sensor_11         0
sensor_12         0
sensor_13         0
sensor_14         0
sensor_15         0
sensor_16         0
sensor_17         0
sensor_18         0
sensor_19         0
sensor_20         0
sensor_21         0
dtype: int64

Check for constant sensor values for removal

In [8]:
suspected_cols = ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
print("Checking unique values for Suspected Columns")
# If output is 1, the value is constant
print(df[suspected_cols].nunique())

Checking unique values for Suspected Columns
op_setting_3    1
sensor_1        1
sensor_5        1
sensor_6        2
sensor_10       1
sensor_16       1
sensor_18       1
sensor_19       1
dtype: int64


Drop those columns from train and test dataset

In [9]:
df.drop(suspected_cols, axis=1, inplace=True)
test_df.drop(suspected_cols, axis=1, inplace=True)

print(f"Constant columns are dropped. New Training Dataset Shape: {df.shape}")

Constant columns are dropped. New Training Dataset Shape: (20631, 18)


# **Target Engineering**

- Calculate Linear RUL: Group your data by engine ID. For each engine, find its maximum number of cycles (when it failed). The true RUL at any given cycle is simply the maximum cycles minus the current cycle.

- Apply Piecewise clipping: Engines don't degrade linearly from day one. An engine at cycle 5 is functionally identical to an engine at cycle 50. Cap your maximum RUL at a specific threshold (usually 125 or 130). Your target variable should look like a flat line at 130 that eventually slopes downward to zero.

In [10]:
# Group data by engine ID (unit_number) & find the failure point
max_cycles = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycles.rename(columns={'time_in_cycles': 'max_cycles'}, inplace=True)
print(max_cycles)

# Merge max_cycles into the main
df = df.merge(max_cycles, on='unit_number', how='left')
df

    unit_number  max_cycles
0             1         192
1             2         287
2             3         179
3             4         189
4             5         269
..          ...         ...
95           96         336
96           97         202
97           98         156
98           99         185
99          100         200

[100 rows x 2 columns]


,unit_number,time_in_cycles,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,max_cycles
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,192
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,192
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,192
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,192
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,643.49,1597.98,1428.63,551.43,2388.19,9065.52,48.07,519.49,2388.26,8137.60,8.4956,397,38.49,22.9735,200
20627,100,197,-0.0016,-0.0005,643.54,1604.50,1433.58,550.86,2388.23,9065.11,48.04,519.68,2388.22,8136.50,8.5139,395,38.30,23.1594,200
20628,100,198,0.0004,0.0000,643.42,1602.46,1428.18,550.94,2388.24,9065.90,48.09,520.01,2388.24,8141.05,8.5646,398,38.44,22.9333,200
20629,100,199,-0.0011,0.0003,643.23,1605.26,1426.53,550.68,2388.25,9073.72,48.39,519.67,2388.23,8139.29,8.5389,395,38.29,23.0640,200


In [11]:
# Calculate the raw linear countdown (Raw RUL)
df['RUL'] = df['max_cycles'] - df['time_in_cycles']

# Cap the values at a certain threshold using .clip() function
df['RUL'] = df['RUL'].clip(upper=130)

# Remove max_cycles (not needed anymore)
df.drop(columns=['max_cycles'], inplace=True)

In [12]:
df

,unit_number,time_in_cycles,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,130
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,130
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,130
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,130
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,643.49,1597.98,1428.63,551.43,2388.19,9065.52,48.07,519.49,2388.26,8137.60,8.4956,397,38.49,22.9735,4
20627,100,197,-0.0016,-0.0005,643.54,1604.50,1433.58,550.86,2388.23,9065.11,48.04,519.68,2388.22,8136.50,8.5139,395,38.30,23.1594,3
20628,100,198,0.0004,0.0000,643.42,1602.46,1428.18,550.94,2388.24,9065.90,48.09,520.01,2388.24,8141.05,8.5646,398,38.44,22.9333,2
20629,100,199,-0.0011,0.0003,643.23,1605.26,1426.53,550.68,2388.25,9073.72,48.39,519.67,2388.23,8139.29,8.5389,395,38.29,23.0640,1


# **Temporal Preprocessing**

- Strict Normalization: Scale your remaining sensors so they share a similar range (e.g., between 0 and 1). Crucial rule: Fit your scaler only on your training set, then apply that exact scaling to your test set.

- Build Sliding Windows:  You cannot feed single, isolated timestamps into your model. You need to extract sequences. Choose a window size (e.g., 30 cycles). Extract cycles 1-30 as your first sample, 2-31 as your second, 3-32 as your third, and so on.

- Tensor Formatting: Reshape this windowed data into a 3D format: (Total Samples, Window Size, Number of Sensors). This format is required for deep learning sequence models.

In [13]:
scaler = MinMaxScaler()

# Define which columns to scale
sensor_cols = ['sensor_2', 'sensor_3','sensor_4','sensor_7','sensor_8','sensor_9','sensor_11','sensor_12','sensor_13','sensor_14','sensor_15','sensor_17','sensor_20','sensor_21']

# Fit the scaler only on training and transform it
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

df

,unit_number,time_in_cycles,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,0.183735,0.406802,0.309757,0.726248,0.242424,0.109755,0.369048,0.633262,0.205882,0.199608,0.363986,0.333333,0.713178,0.724662,130
1,1,2,0.0019,-0.0003,0.283133,0.453019,0.352633,0.628019,0.212121,0.100242,0.380952,0.765458,0.279412,0.162813,0.411312,0.333333,0.666667,0.731014,130
2,1,3,-0.0043,0.0003,0.343373,0.369523,0.370527,0.710145,0.272727,0.140043,0.250000,0.795309,0.220588,0.171793,0.357445,0.166667,0.627907,0.621375,130
3,1,4,0.0007,0.0000,0.343373,0.256159,0.331195,0.740741,0.318182,0.124518,0.166667,0.889126,0.294118,0.174889,0.166603,0.333333,0.573643,0.662386,130
4,1,5,-0.0019,-0.0002,0.349398,0.257467,0.404625,0.668277,0.242424,0.149960,0.255952,0.746269,0.235294,0.174734,0.402078,0.416667,0.589147,0.704502,130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,0.686747,0.587312,0.782917,0.254428,0.439394,0.196491,0.726190,0.170576,0.558824,0.194344,0.656791,0.750000,0.271318,0.109500,4
20627,100,197,-0.0016,-0.0005,0.701807,0.729453,0.866475,0.162641,0.500000,0.194651,0.708333,0.211087,0.500000,0.188668,0.727203,0.583333,0.124031,0.366197,3
20628,100,198,0.0004,0.0000,0.665663,0.684979,0.775321,0.175523,0.515152,0.198196,0.738095,0.281450,0.529412,0.212148,0.922278,0.833333,0.232558,0.053991,2
20629,100,199,-0.0011,0.0003,0.608434,0.746021,0.747468,0.133655,0.530303,0.233285,0.916667,0.208955,0.514706,0.203065,0.823394,0.583333,0.116279,0.234466,1


In [14]:
def create_sliding_window(df, window_size, feature_cols, target_col):
    """
    Transform 2D tabular data into 3D tensors (Important for Neural Networks)
    """
    X_windows = []
    y_targets = []
    
    # Isolate each engine so windows don't overlap between different engines
    for engine_id, engine_data in df.groupby('unit_number'):
        
        # Convert pandas columns to fast numpy arrays
        engine_features = engine_data[feature_cols].values
        engine_target = engine_data[target_col].values
        
        # Slide the window down the engine's timeline step by step
        for i in range(len(engine_data) - window_size + 1):
            
            # Extract the 2D window
            window = engine_features[i : i + window_size, :]
            X_windows.append(window)
            
            # The target is the RUL at the VERY LAST time step of this specific window
            target = engine_target[i + window_size - 1]
            y_targets.append(target)
    
    return np.array(X_windows), np.array(y_targets)

In [15]:
X_train, y_train = create_sliding_window(df=df, window_size=30, feature_cols=sensor_cols, target_col='RUL')
print(f"Original 2D DataFrame shape: {df.shape}")
print(f"New 3D Training Tensor shape: {X_train.shape}")

Original 2D DataFrame shape: (20631, 19)
New 3D Training Tensor shape: (17731, 30, 14)


# **Model Training**

- Baseline Model Testing (Random Forest & XGBoost)
- Train test split (test data already available)

## XGBoost

In [16]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth = 5,
    objective = 'reg:squarederror',
    random_state = 42,
    n_jobs = -1
)

# Flatten the 3D data into 2D
# Extract the the 3D dimension of the training data
num_samples, window_size, num_sensors = X_train.shape

# Reshape the 3D array into a 2D array
X_train_2d = X_train.reshape(num_samples, window_size * num_sensors)

print("Training on XGBoost Regressor as a baseline model....")
xgb.fit(X_train_2d, y_train)
y_train_pred = xgb.predict(X_train_2d)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
print(f"Training RMSE: {train_rmse:.4f} cycles.")

Training on XGBoost Regressor as a baseline model....
Training RMSE: 8.9490 cycles.


This means that ~12 flight can be completed before a failure

In [17]:
def create_test_sliding_window(df, window_size, feature_cols):
    """
    Transform 2D tabular data into 3D tensors (Important for Neural Networks)
    """
    X_test_windows = []
    
    # Isolate each engine so windows don't overlap between different engines
    for engine_id, engine_data in df.groupby('unit_number'):
        # Convert pandas columns to fast numpy arrays
        engine_features = engine_data[feature_cols].values
        
        # Slide the window down the engine's timeline step by step
        if len(engine_features) < window_size:
            
            pad_size = window_size - len(engine_features)
            # Pad the beginning with the first recorded sesor values
            pad_array = np.tile(engine_features[0], (pad_size, 1))
            # Extract the 2D window
            engine_features = np.vstack([pad_array, engine_features])
            
        final_window = engine_features[-window_size:, :]
        X_test_windows.append(final_window)
    
    return np.array(X_test_windows)

In [18]:
test_df[sensor_cols] = scaler.transform(test_df[sensor_cols])

X_test = create_test_sliding_window(df=test_df, window_size=30, feature_cols=sensor_cols)

print(f"Original 2D DataFrame shape: {test_df.shape}")
print(f"New 3D Testing Tensor shape: {X_test.shape}")

Original 2D DataFrame shape: (13096, 18)
New 3D Testing Tensor shape: (100, 30, 14)


In [23]:
# Read the separate RUL file
file_path_test = os.path.join(path, "RUL_FD001.txt")
y_test_df = pd.read_csv(file_path_test, sep=r'\s+', header=None)

# Convert the pd dataFrame into a flat 1D numpy array
y_test = y_test_df.values.flatten()

In [24]:
# Flatten the 3D data into 2D
# Extract the the 3D dimension of the training data
num_samples, window_size, num_sensors = X_test.shape

# Reshape the 3D array into a 2D array
X_test_2d = X_test.reshape(num_samples, window_size * num_sensors)

print("Testing on XGBoost Regressor as a baseline model....")
y_test_pred = xgb.predict(X_test_2d)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
print(f"Testing RMSE: {test_rmse:.4f} cycles.")

Testing on XGBoost Regressor as a baseline model....
Testing RMSE: 14.8676 cycles.
